Основные цели:

1. Понять, какие факторы связаны с наличием диабета
2. Сравнить профиль пациентов с диабетом и без диабета
3. Выделить группы повышенного риска
4. Потренироваться в анализе медицинских данных с помощью Polars

Основные вопросы:

1. Как распределены ключевые признаки в датасете?
2. Как различаются пациенты с диабетом и без диабета?
3. Какие признаки сильнее всего связаны с наличием диабета?
4. Как меняется доля диабета в разных группах пациентов?
5. Какие медицинские и поведенческие факторы могут быть связаны с повышенным риском диабета?
6. Можно ли выделить сегменты пациентов с более высокой вероятностью диабета?
7. Есть ли в данных аномалии, выбросы или подозрительные значения?
8. Какие признаки дают наибольший вклад в описание группы повышенного риска?

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# https://huggingface.co/datasets/marianeft/diabetes_prediction_dataset

import polars as pl


In [3]:
df = pl.read_csv('/content/drive/MyDrive/Colab Notebooks/diabetes_prediction_dataset.csv')

# Структура
print(df.head())
# Размерность
print(df.shape)
# Столбцы + типы данных
print(df.schema)
# Статистическая сводка
print(df.describe())
# Кол-во нулевых значений
print(df.null_count())

shape: (5, 9)
┌────────┬──────┬──────────────┬───────────────┬───┬───────┬─────────────┬──────────────┬──────────┐
│ gender ┆ age  ┆ hypertension ┆ heart_disease ┆ … ┆ bmi   ┆ HbA1c_level ┆ blood_glucos ┆ diabetes │
│ ---    ┆ ---  ┆ ---          ┆ ---           ┆   ┆ ---   ┆ ---         ┆ e_level      ┆ ---      │
│ str    ┆ f64  ┆ i64          ┆ i64           ┆   ┆ f64   ┆ f64         ┆ ---          ┆ i64      │
│        ┆      ┆              ┆               ┆   ┆       ┆             ┆ i64          ┆          │
╞════════╪══════╪══════════════╪═══════════════╪═══╪═══════╪═════════════╪══════════════╪══════════╡
│ Female ┆ 80.0 ┆ 0            ┆ 1             ┆ … ┆ 25.19 ┆ 6.6         ┆ 140          ┆ 0        │
│ Female ┆ 54.0 ┆ 0            ┆ 0             ┆ … ┆ 27.32 ┆ 6.6         ┆ 80           ┆ 0        │
│ Male   ┆ 28.0 ┆ 0            ┆ 0             ┆ … ┆ 27.32 ┆ 5.7         ┆ 158          ┆ 0        │
│ Female ┆ 36.0 ┆ 0            ┆ 0             ┆ … ┆ 23.45 ┆ 5.0         ┆ 15

In [4]:
# Кол-во никальных значений
print(df.n_unique())
print(df.unique())

96146
shape: (96_146, 9)
┌────────┬──────┬──────────────┬───────────────┬───┬───────┬─────────────┬──────────────┬──────────┐
│ gender ┆ age  ┆ hypertension ┆ heart_disease ┆ … ┆ bmi   ┆ HbA1c_level ┆ blood_glucos ┆ diabetes │
│ ---    ┆ ---  ┆ ---          ┆ ---           ┆   ┆ ---   ┆ ---         ┆ e_level      ┆ ---      │
│ str    ┆ f64  ┆ i64          ┆ i64           ┆   ┆ f64   ┆ f64         ┆ ---          ┆ i64      │
│        ┆      ┆              ┆               ┆   ┆       ┆             ┆ i64          ┆          │
╞════════╪══════╪══════════════╪═══════════════╪═══╪═══════╪═════════════╪══════════════╪══════════╡
│ Male   ┆ 12.0 ┆ 0            ┆ 0             ┆ … ┆ 19.04 ┆ 6.6         ┆ 85           ┆ 0        │
│ Female ┆ 58.0 ┆ 0            ┆ 0             ┆ … ┆ 27.32 ┆ 5.8         ┆ 140          ┆ 0        │
│ Male   ┆ 38.0 ┆ 0            ┆ 0             ┆ … ┆ 24.01 ┆ 5.8         ┆ 130          ┆ 0        │
│ Female ┆ 1.32 ┆ 0            ┆ 0             ┆ … ┆ 22.28 ┆ 5.8  

In [5]:
# Список имен столбцов
print(df.columns)

['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']


# Проверка корректности значений.

In [6]:
print('Пол')
count_values_gender = df['gender'].value_counts()
print(count_values_gender)
count_strange_gender = count_values_gender.filter(pl.col("gender") == 'Other')['count'][0] * 100 / df.shape[0]
print(f"Процент other: {count_strange_gender}")
print('Возраст')
count_strange_age = df.filter(pl.col("age") < 1).count()['age'][0] * 100 / df.shape[0]
print(f"Процент людей с возрастом <1 года: {count_strange_age}")
print('Гипертония')
count_values_hypertension = df['hypertension'].value_counts()
print(count_values_hypertension)
print('Сердечно-сосудистое заболевание')
count_values_heart_disease = df['heart_disease'].value_counts()
print(count_values_heart_disease)
print('История курения')
count_values_smoking_history = df['smoking_history'].value_counts()
print(count_values_smoking_history)
print('Индекс массы тела')
count_values_bmi = df['bmi'].value_counts()
print(count_values_bmi)
# Уровень гликированного гемоглобина в крови
# < 5.7% - норма
# 5.7% - 6.4% - предиабет
# >= 6.5% - возможный диабет
print('Уровень гликированного гемоглобина в крови')
count_values_HbA1c_level = df['HbA1c_level'].value_counts()
print(count_values_HbA1c_level)
print('Уровень глюкозы в крови')
count_values_blood_glucose_level = df['blood_glucose_level'].value_counts()
print(count_values_blood_glucose_level)
print('Наличие диабета')
count_values_diabetes = df['diabetes'].value_counts()
print(count_values_diabetes)

Пол
shape: (3, 2)
┌────────┬───────┐
│ gender ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ Female ┆ 58552 │
│ Male   ┆ 41430 │
│ Other  ┆ 18    │
└────────┴───────┘
Процент other: 0.018
Возраст
Процент людей с возрастом <1 года: 0.911
Гипертония
shape: (2, 2)
┌──────────────┬───────┐
│ hypertension ┆ count │
│ ---          ┆ ---   │
│ i64          ┆ u32   │
╞══════════════╪═══════╡
│ 0            ┆ 92515 │
│ 1            ┆ 7485  │
└──────────────┴───────┘
Сердечно-сосудистое заболевание
shape: (2, 2)
┌───────────────┬───────┐
│ heart_disease ┆ count │
│ ---           ┆ ---   │
│ i64           ┆ u32   │
╞═══════════════╪═══════╡
│ 0             ┆ 96058 │
│ 1             ┆ 3942  │
└───────────────┴───────┘
История курения
shape: (6, 2)
┌─────────────────┬───────┐
│ smoking_history ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ former          ┆ 9352  │
│ not current     ┆ 6447  │
│ No Info         ┆ 35816 │
│ curren

# Обработка, удаление и фильтрация подозрительных значений.

In [7]:
# Удаление строк с gender=Other
df = df.filter(pl.col("gender") != "Other")

In [8]:
# Удаление подозрительного возраста <1 года
df = df.filter(pl.col("age") > 1)

In [9]:
# Заменить значения в столбце smoking_history
dct_ru = {'No Info': 'Нет информации',
          'not current': 'В настоящее время не курит',
          'ever': 'Курил(а) когда-либо',
          'former': 'Бывший курильщик',
          'current': 'Курит сейчас',
          'never': 'Никогда не курил(а)'}
df = df.with_columns(
    pl.col("smoking_history").replace(dct_ru)
)
df['smoking_history'].value_counts()

smoking_history,count
str,u32
"""Нет информации""",34867
"""Бывший курильщик""",9352
"""Курит сейчас""",9285
"""Курил(а) когда-либо""",4003
"""Никогда не курил(а)""",35051
"""В настоящее время не курит""",6430


In [10]:
# Странные единицы измерения. Перевод mg/dL в ммоль/л
# mmol_l = mg_dl / 18
df = df.with_columns(
    (pl.col("blood_glucose_level") / 18)
    .round(3)
    .alias("blood_glucose_level")
)

## Подготовка категориальных данных.

In [11]:
# Возраст
df = df.with_columns(
    pl.when(pl.col("age") < 20).then(pl.lit("<20"))
    .when(pl.col("age") < 30).then(pl.lit("20-30"))
    .when(pl.col("age") < 40).then(pl.lit("30-40"))
    .when(pl.col("age") < 50).then(pl.lit("40-50"))
    .when(pl.col("age") < 60).then(pl.lit("50-60"))
    .when(pl.col("age") < 70).then(pl.lit("60-70"))
    .otherwise(pl.lit("70+"))
    .alias("age_group")
)

In [12]:
# Индекс массы тела
df = df.with_columns(
    pl.when(pl.col("bmi") < 20).then(pl.lit("<20"))
    .when(pl.col("bmi") < 30).then(pl.lit("20-30"))
    .when(pl.col("bmi") < 40).then(pl.lit("30-40"))
    .when(pl.col("bmi") < 50).then(pl.lit("40-50"))
    .when(pl.col("bmi") < 60).then(pl.lit("50-60"))
    .when(pl.col("bmi") < 70).then(pl.lit("60-70"))
    .when(pl.col("bmi") < 80).then(pl.lit("70-80"))
    .when(pl.col("bmi") < 90).then(pl.lit("80-90"))
    .otherwise(pl.lit("90+"))
    .alias("bmi_group")
)

In [13]:
# Уровень гликированного гемоглобина в крови
df = df.with_columns(
    pl.when(pl.col("HbA1c_level") < 4).then(pl.lit("<4"))
    .when(pl.col("HbA1c_level") < 5).then(pl.lit("4-5"))
    .when(pl.col("HbA1c_level") < 6).then(pl.lit("5-6"))
    .when(pl.col("HbA1c_level") < 7).then(pl.lit("6-7"))
    .when(pl.col("HbA1c_level") < 8).then(pl.lit("7-8"))
    .otherwise(pl.lit(">=8"))
    .alias("HbA1c_level_group")
)

In [14]:
# Уровень гликированного гемоглобина в крови
df = df.with_columns(
    pl.when(pl.col("blood_glucose_level") < 5.6).then(pl.lit("<5.6"))
    .when(pl.col("blood_glucose_level") < 7.8).then(pl.lit("5.6-7.8"))
    .when(pl.col("blood_glucose_level") < 10.0).then(pl.lit("7.8-10.0"))
    .when(pl.col("blood_glucose_level") < 12.2).then(pl.lit("10.0-12.2"))
    .otherwise(pl.lit("12.2+"))
    .alias("blood_glucose_group")
)

## Базовый EDA

In [15]:
# Статистическая агрегация для столбца diabetes
df.group_by("diabetes").agg(
    pl.len().alias("cnt"),
    pl.col("age").mean().alias("avg_age"),
    pl.col("bmi").mean().alias("avg_bmi"),
    pl.col("HbA1c_level").mean().alias("avg_hba1c"),
    pl.col("blood_glucose_level").mean().alias("avg_glucose")
)

diabetes,cnt,avg_age,avg_bmi,avg_hba1c,avg_glucose
i64,u32,f64,f64,f64,f64
1,8500,60.946588,31.988382,6.934953,10.783029
0,90488,40.551718,26.991022,5.397158,7.380887


Вывод: в среднем люди с диабетом имеют возраст - 60 лет, индекс массы тела - 31, уровень гликированного гемоглобина в крови - 6,9, уровень глюкозы - 10,7.

In [16]:
# Вывод статистик отдельно для diabetes=1
group_by_diabetes = (
    df.filter(pl.col("diabetes") == 1).group_by("diabetes")
    .agg(pl.col("age").mean().alias("avg_age"),
         pl.col("bmi").mean().alias("avg_bmi"),
         pl.col("HbA1c_level").mean().alias("avg_hba1c"),
         pl.col("blood_glucose_level").mean().alias("avg_glucose"))
)
group_by_diabetes

diabetes,avg_age,avg_bmi,avg_hba1c,avg_glucose
i64,f64,f64,f64,f64
1,60.946588,31.988382,6.934953,10.783029


In [17]:
df.columns

['gender',
 'age',
 'hypertension',
 'heart_disease',
 'smoking_history',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'diabetes',
 'age_group',
 'bmi_group',
 'HbA1c_level_group',
 'blood_glucose_group']

## Доля диабета по разным категориям:

In [18]:
# Возраст
df.group_by("age_group").agg(
    pl.len().alias("cnt"),
    pl.col("diabetes").mean().alias("diabetes_rate")
).sort("diabetes_rate", descending=True)

age_group,cnt,diabetes_rate
str,u32,f64
"""70+""",13275,0.201356
"""60-70""",11780,0.191766
"""50-60""",14860,0.125101
"""40-50""",14592,0.068462
"""30-40""",13051,0.033407
"""20-30""",12763,0.013163
"""<20""",18667,0.005678


In [19]:
# Индекс массы тела
df.group_by("bmi_group").agg(
    pl.len().alias("cnt"),
    pl.col("diabetes").mean().alias("diabetes_rate")
).sort("diabetes_rate", descending=True)

bmi_group,cnt,diabetes_rate
str,u32,f64
"""80-90""",6,0.5
"""60-70""",97,0.319588
"""70-80""",10,0.3
"""50-60""",668,0.290419
"""40-50""",3837,0.251238
"""30-40""",18906,0.16069
"""20-30""",64198,0.064441
"""<20""",11263,0.011542
"""90+""",3,0.0


In [20]:
# Уровень гликированного гемоглобина в крови
df.group_by("HbA1c_level_group").agg(
    pl.len().alias("cnt"),
    pl.col("diabetes").mean().alias("diabetes_rate")
).sort("diabetes_rate", descending=True)

HbA1c_level_group,cnt,diabetes_rate
str,u32,f64
""">=8""",1976,1.0
"""7-8""",1277,1.0
"""6-7""",41761,0.093005
"""5-6""",23939,0.056936
"""<4""",7560,0.0
"""4-5""",22475,0.0


## Топ риск-сегментов

In [21]:
# Доля диабета по всему датасету
base_rate = df.select(pl.col("diabetes").mean()).item()
print(f"Общая доля диабета в датасете: {base_rate:.4f} ({base_rate:.2%})")

Общая доля диабета в датасете: 0.0859 (8.59%)


In [22]:
top_risk_segments = (
    df
    .group_by(["age_group", "bmi_group", "hypertension", "heart_disease"])
    .agg(
        pl.len().alias("cnt"),
        pl.col("diabetes").mean().alias("diabetes_rate"),
        pl.col("HbA1c_level").mean().alias("avg_hba1c"),
        pl.col("blood_glucose_level").mean().alias("avg_glucose"),
        pl.col("bmi").mean().alias("avg_bmi")
    )
    .filter(pl.col("cnt") >= 30)
    .with_columns((pl.col("diabetes_rate") - base_rate).alias("rate_diff_vs_total"))
    .sort("diabetes_rate", descending=True)
)

top_risk_segments.head(10)

age_group,bmi_group,hypertension,heart_disease,cnt,diabetes_rate,avg_hba1c,avg_glucose,avg_bmi,rate_diff_vs_total
str,str,i64,i64,u32,f64,f64,f64,f64,f64
"""70+""","""40-50""",0,1,39,0.641026,6.187179,9.457256,43.084615,0.555157
"""70+""","""40-50""",1,0,61,0.622951,6.37541,9.025557,43.08,0.537082
"""60-70""","""40-50""",0,1,55,0.618182,6.174545,9.028255,43.228727,0.532313
"""60-70""","""40-50""",1,0,128,0.53125,6.3203125,9.441828,43.750625,0.445381
"""50-60""","""40-50""",0,1,33,0.515152,6.172727,8.841818,43.694545,0.429283
"""50-60""","""30-40""",1,1,59,0.491525,6.294915,8.878475,34.007797,0.405656
"""60-70""","""30-40""",0,1,240,0.479167,6.1425,9.11595,33.655083,0.393298
"""60-70""","""50-60""",0,0,69,0.478261,6.253623,9.649739,53.367971,0.392392
"""70+""","""30-40""",1,1,122,0.45082,6.129508,8.501844,33.402131,0.364951


Первичный вывод по этому сегменту:
1. Сегментный анализ показывает, что риск диабета максимален у пациентов старшего возраста с высоким BMI, особенно при наличии гипертонии или сердечно-сосудистых заболеваний.
2. Наиболее выраженный профиль повышенного риска в датасете формируется сочетанием возраста, ожирения и сопутствующих хронических состояний.

In [23]:
# Корреляция между диабетом, полом, возрастом и курением
risk_segments_behavior = (
    df
    .group_by(["gender", "age_group", "smoking_history"])
    .agg(
        pl.len().alias("cnt"),
        pl.col("diabetes").mean().alias("diabetes_rate"),
        pl.col("bmi").mean().alias("avg_bmi"),
        pl.col("HbA1c_level").mean().alias("avg_hba1c"),
        pl.col("blood_glucose_level").mean().alias("avg_glucose")
    )
    .filter(pl.col("cnt") >= 30)
    .with_columns((pl.col("diabetes_rate") - base_rate).alias("rate_diff_vs_total"))
    .sort("diabetes_rate", descending=True)
)

risk_segments_behavior.head(10)

gender,age_group,smoking_history,cnt,diabetes_rate,avg_bmi,avg_hba1c,avg_glucose,rate_diff_vs_total
str,str,str,u32,f64,f64,f64,f64,f64
"""Male""","""70+""","""Бывший курильщик""",1373,0.281136,28.298165,5.789803,8.344307,0.195267
"""Male""","""70+""","""Никогда не курил(а)""",1323,0.270597,28.055941,5.790779,8.316073,0.184728
"""Male""","""60-70""","""Бывший курильщик""",1120,0.267857,30.306527,5.840536,8.318357,0.181988
"""Male""","""70+""","""Курит сейчас""",284,0.260563,27.164648,5.852465,8.459511,0.174694
"""Male""","""60-70""","""Никогда не курил(а)""",1428,0.25,29.897213,5.841036,8.257586,0.164131
"""Male""","""60-70""","""Курит сейчас""",533,0.249531,28.474934,5.709006,8.359064,0.163662
"""Male""","""60-70""","""Курил(а) когда-либо""",315,0.244444,29.603333,5.790159,8.216575,0.158575
"""Male""","""70+""","""Курил(а) когда-либо""",308,0.243506,27.16013,5.76039,8.205834,0.157637
"""Female""","""70+""","""Курит сейчас""",300,0.23,25.987167,5.675,7.963703,0.144131


Первичный вывод по этому сегменту:
1. Наибольшая доля диабета наблюдается у мужчин 60+ и 70+.
2. Возраст в этом срезе влияет на риск сильнее, чем история курения.
3. У топ-сегментов также выше средние BMI, HbA1c и glucose.

In [24]:
# Корреляция между диабетом, HbA1c, glucose и возрастом
risk_segments_lab = (
    df
    .group_by(["age_group", "HbA1c_level_group", "blood_glucose_group"])
    .agg(
        pl.len().alias("cnt"),
        pl.col("diabetes").mean().alias("diabetes_rate"),
        pl.col("bmi").mean().alias("avg_bmi"),
        pl.col("hypertension").mean().alias("hypertension_rate"),
        pl.col("heart_disease").mean().alias("heart_disease_rate"),
        pl.col("blood_glucose_level").mean().alias("avg_glucose_mmol_l")
    )
    .filter(pl.col("cnt") >= 30)
    .with_columns(
        (pl.col("diabetes_rate") - base_rate).alias("rate_diff_vs_total")
    )
    .sort("diabetes_rate", descending=True)
)

risk_segments_lab.head(10)

age_group,HbA1c_level_group,blood_glucose_group,cnt,diabetes_rate,avg_bmi,hypertension_rate,heart_disease_rate,avg_glucose_mmol_l,rate_diff_vs_total
str,str,str,u32,f64,f64,f64,f64,f64,f64
"""30-40""",""">=8""","""5.6-7.8""",32,1.0,35.410938,0.125,0.0,7.3194375,0.914131
"""50-60""","""7-8""","""7.8-10.0""",82,1.0,35.41122,0.280488,0.109756,8.645646,0.914131
"""40-50""","""7-8""","""5.6-7.8""",46,1.0,36.952391,0.173913,0.065217,7.335739,0.914131
"""50-60""",""">=8""","""10.0-12.2""",34,1.0,31.154412,0.205882,0.058824,11.111,0.914131
"""70+""","""7-8""","""12.2+""",154,1.0,29.610844,0.324675,0.24026,14.516558,0.914131
"""60-70""","""5-6""","""12.2+""",156,1.0,32.925705,0.262821,0.179487,14.786327,0.914131
"""40-50""","""7-8""","""7.8-10.0""",51,1.0,34.186667,0.196078,0.078431,8.659059,0.914131
"""60-70""",""">=8""","""12.2+""",215,1.0,33.076186,0.251163,0.176744,14.39786,0.914131
"""50-60""",""">=8""","""5.6-7.8""",90,1.0,32.395111,0.288889,0.111111,7.333333,0.914131


Первичный вывод по этому сегменту:
1. Сильный фактор риска - это высокие значения HbA1c и glucose(достигает 100%)
2. Возраст и BMI дополнительно усиливают риск.

In [25]:
# Корреляция между диабетом, BMI, HbA1c и гипертонией
risk_segments_physiology = (
    df
    .group_by(["bmi_group", "HbA1c_level_group", "hypertension"])
    .agg(
        pl.len().alias("cnt"),
        pl.col("diabetes").mean().alias("diabetes_rate"),
        pl.col("blood_glucose_level").mean().alias("avg_glucose"),
        pl.col("age").mean().alias("avg_age"),
        pl.col("heart_disease").mean().alias("heart_disease_rate")
    )
    .filter(pl.col("cnt") >= 30)
    .with_columns(
        (pl.col("diabetes_rate") - base_rate).alias("rate_diff_vs_total")
    )
    .sort("diabetes_rate", descending=True)
)

risk_segments_physiology.head(10)

bmi_group,HbA1c_level_group,hypertension,cnt,diabetes_rate,avg_glucose,avg_age,heart_disease_rate,rate_diff_vs_total
str,str,i64,u32,f64,f64,f64,f64,f64
"""20-30""",""">=8""",1,218,1.0,10.431702,67.307339,0.201835,0.914131
"""30-40""","""7-8""",0,307,1.0,10.919466,59.863192,0.140065,0.914131
"""50-60""",""">=8""",0,31,1.0,10.232935,53.709677,0.0,0.914131
"""40-50""","""7-8""",1,49,1.0,10.899041,59.979592,0.122449,0.914131
"""30-40""",""">=8""",1,182,1.0,10.964885,61.895604,0.17033,0.914131
"""20-30""","""7-8""",0,469,1.0,10.822527,61.238806,0.15565,0.914131
"""40-50""",""">=8""",1,65,1.0,11.441138,60.184615,0.138462,0.914131
"""40-50""",""">=8""",0,137,1.0,10.56208,55.912409,0.124088,0.914131
"""40-50""","""7-8""",0,116,1.0,10.553138,56.896552,0.086207,0.914131


Первичный вывод по этому сегменту:
1. Главный фактор в этом сегменте - это высокий HbA1c и высокий glucose(BMI влияет меньше)

## Финальный вывод по проекту:
1. Диабет связан с комбинацией факторов, а не с одним признаком.
2. Наиболее важные факторы риска: возраст, высокий индекс массы тела, уровень гликированного гемоглобина и уровень глюкозы в крови.
3. Гипертония и сердечные заболевания дополнительно усиливают риск.
4. Пол и курение влияют слабее, чем возраст и метаболические показатели.
5. Итог по пациентам: наиболее выраженный профиль риска - это пожилые пациенты с ожирением и ухудшенными лабораторными показателями.